In [ ]:
!pip install sentence-transformers
!pip install nltk
!pip install scikit-learn
!pip install pandas
!pip install numpy

In [ ]:
import pandas as pd
import numpy as np
import nltk
import re
import string

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import MinMaxScaler

from sentence_transformers import SentenceTransformer

nltk.download('stopwords')
nltk.download('wordnet')

print("Libraries loaded successfully")

# Upload Resume Dataset


In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
df = pd.read_csv("Resume.csv")

print("Dataset loaded")
df.head()

Cleaning the Resume txt File

In [ ]:

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))

    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]

    return " ".join(words)

df['cleaned_resume'] = df['Resume_str'].apply(preprocess_text)

print("Text preprocessing completed")
df.head()

# Load BERT Model

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

print("BERT model loaded")

In [ ]:
# Convert Resume to Embeddings
resume_embeddings = model.encode(
    df['cleaned_resume'].tolist(),
    show_progress_bar=True
)

print("Embeddings created")

In [ ]:
# Skill Extraction Score
skills = [
    "python","machine learning","deep learning","nlp",
    "sql","tensorflow","pytorch","data analysis",
    "excel","communication","leadership"
]

def skill_score(text):
    score = 0
    for skill in skills:
        if skill in text:
            score += 1
    return score

df['skill_score'] = df['cleaned_resume'].apply(skill_score)

print("Skill scoring completed")

In [ ]:
# Cluster Resumes
kmeans = KMeans(n_clusters=8, random_state=42)

df['cluster'] = kmeans.fit_predict(resume_embeddings)

score = silhouette_score(resume_embeddings, df['cluster'])

print("Clustering completed")
print("Silhouette Score:", score)

In [ ]:
# Enter Job Description
job_description = """
Looking for Data Scientist with Python, Machine Learning,
Deep Learning, NLP, SQL, TensorFlow experience
"""

cleaned_jd = preprocess_text(job_description)

jd_embedding = model.encode([cleaned_jd])

similarity = cosine_similarity(jd_embedding, resume_embeddings)[0]

df['similarity_score'] = similarity

print("Similarity calculated")

In [ ]:
# Calculate Final Score (0–10 Ranking)
scaler = MinMaxScaler()

df['similarity_score'] = scaler.fit_transform(df[['similarity_score']])
df['skill_score'] = scaler.fit_transform(df[['skill_score']])

df['final_score'] = (0.7 * df['similarity_score'] +
                     0.3 * df['skill_score']) * 10

print("Final scoring completed")

In [ ]:
# Show Top Candidates
top_candidates = df.sort_values(by='final_score', ascending=False)

top_candidates[['Category','final_score']].head(10)

In [ ]:
df.to_csv("ranked_resumes.csv", index=False)

print("File saved")

files.download("ranked_resumes.csv")

In [ ]:
import json

try:
    with open("AI_Resume_Parser.ipynb", "r") as f:
        nb = json.load(f)

    if "widgets" in nb["metadata"]:
        del nb["metadata"]["widgets"]

    with open("AI_Resume_Parser_clean.ipynb", "w") as f:
        json.dump(nb, f)

    print("Clean notebook created")
except FileNotFoundError:
    print("Error: 'AI_Resume_Parser.ipynb' not found. Please ensure the notebook file is uploaded to your Colab environment.")